## Cruzamento dos Dados (Enunciado Questões + Microdados)

* **Input**: Local do CSV de Questões Extraídas + Local do CSV de Microdados

* **Output**: CSV com as colunas -> *numero_questao*, *enunciado*, *alternativas*, *nu_param_B* e *gabarito*

In [3]:
import pandas as pd

In [69]:
# Definição de Variáveis de Input e Output
flag_ledor=1
year = "2023"
if flag_ledor:
    year+="_LEDOR"

question_path = f"../data/extracted_questions/enem_{year}.csv"
microdados_path = f"../data/microdados/microdados_{year[:4]}.csv"
output_path = f"../data/merged_data/enem_{year}.csv"

In [70]:
# Ler o CSV extraído do PDF (resultado de Text_Extraction.ipynb)
df_questions = pd.read_csv(question_path, encoding="utf-8", quotechar='"')

# Ler o CSV de microdados
df_microdados = pd.read_csv(microdados_path, delimiter=";", encoding="latin1")

In [71]:
co_prova = {
    "2015": 275,
    "2016": 351,
    "2017": 391,
    "2018_LEDOR": 463,
    "2019_LEDOR": 519,
    "2020": 597,
    "2020_LEDOR": 604,
    "2021_LEDOR": 916,
    "2022_LEDOR": 1092,
    "2023_LEDOR": 1228,
    "2009": 49,
    "2010": 89,
    "2011": 121
}

In [72]:
# Filtrando os microdados para SG_AREA 'CN', TX_COR 'AZUL' e CO_PROVA 'X'
# ATENÇÃO: O Código da Prova para a Aplicação Regular muda a cada ano
df_microdados_filtrado = df_microdados[
    (df_microdados["SG_AREA"] == "CN")  # Ciência das Naturezas
    & (
        (df_microdados["TX_COR"].str.upper() == "LARANJA" if flag_ledor else "AZUL") 
    )  # Apenas caderno azul
    & (df_microdados["CO_PROVA"] == co_prova[year])  # Aplicação Regular
]

# Selecionar apenas as colunas de interesse dos microdados
df_microdados_sel = df_microdados_filtrado[
    ["CO_POSICAO", "NU_PARAM_B", "TX_GABARITO"]
].copy()

In [73]:
# Certificar que os tipos das chaves de junção são compatíveis
df_questions["numero_questao"] = df_questions["numero_questao"].astype(str)
df_microdados_sel["CO_POSICAO"] = df_microdados_sel["CO_POSICAO"].astype(str)

# Fazer o cruzamento usando o número da questão (numero_questao e CO_POSICAO)
df_merged = pd.merge(
    df_questions,
    df_microdados_sel,
    left_on="numero_questao",
    right_on="CO_POSICAO",
    how="left",
)

In [74]:
# Remover a coluna CO_POSICAO (redundante)
df_merged.drop(columns=["CO_POSICAO"], inplace=True)

# Renomeando colunas
df_merged.rename(columns={
    "NU_PARAM_B": "nu_param_B",
    "TX_GABARITO": "gabarito"
}, inplace=True)


# Salvando DF resultante
df_merged.to_csv(output_path, index=False, encoding="utf-8")

print("Merge concluído.")

Merge concluído.


In [75]:
df = pd.read_csv(output_path)
df.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito
0,91,É comum em viagens de avião sermos solicitados...,A: forem ambas audíveis.; B: tiverem a mesma p...,-0.45616,C
1,92,Terpenos com atividade inseticida: \numa alte...,"A: Saliva, por consequência da atividade de en...",1.60649,E
2,93,Cafeteria adota copo reutilizável \nfeito com ...,A: Ter a durabilidade de uma cerâmica e ser to...,0.32873,D
3,94,Um método simples para avaliar o teor \nde sac...,A: maior densidade.; B: menor viscosidade.; C:...,0.49481,A
4,95,O descarte de detergentes comuns nos esgotos \...,A: Retirar a parte polar da molécula.; B: Elim...,1.43988,E
